We start with Wages Case and lin reg

In [1]:
# Import required packages 
import pandas as pd
import numpy as np

#Scikit
from sklearn.linear_model import LinearRegression #for lin reg
from sklearn.metrics import r2_score

import statsmodels.api as sm

import os

targetDirectory = "C:\\CASES\\PYTPRA"
os.chdir(targetDirectory)
print("FRANK Working directory in use: ", os. getcwd())

FRANK Working directory in use:  C:\CASES\PYTPRA


In [2]:
### data is looked for in wd
df_WAGES = pd.read_csv('wages.txt', sep = "\t") #txt is tab separated, data from Verbeek, M. "A Guide to Modern Econometrics"
df_WAGES.head()

,EXPER,MALE,SCHOOL,WAGE
0,9,0,13,6.315296
1,12,0,12,5.479770
2,11,0,11,3.642170
3,9,0,14,4.593337
4,8,0,14,2.418157


In [3]:
#behave as if we were filling the world = we excute a data generating process
school = df_WAGES['SCHOOL']
school[2] #offset is zero, hence 3rd row and column SCHOOL
exper = df_WAGES['EXPER']
male = df_WAGES['MALE']

#scikit needs matrix of size n_samples, n_features (van der Plas, p 349)
print(school.shape)
a_school = np.array(school, dtype='float32')
a_school = a_school.reshape(school.size,1)
print(a_school.shape)
a_exper = np.array(exper, dtype='float32')
a_exper = a_exper.reshape(exper.size,1)
a_male = np.array(male, dtype='float32')
a_male = a_male.reshape(male.size,1)
print(a_male.shape)

#head of HR determines wages according to the following TOP SECRET FORMULA
a_wage_gen = -3.38 + 0.63 * a_school + 0.12 * a_exper + 1.34 * a_male  
print("First five wages are set as: \n", a_wage_gen[0:5,:])

#to hide that discrimination happens, a bit of noise is added to this equation
np.random.seed(2023)#make it reproducible
a_eps = np.random.normal(loc=0.0, scale=0.2, size=school.size)
a_eps = a_eps.reshape(school.size,1)
print("Mean of error term e:", round(a_eps.mean(),3))
print("Maximum of error term e:", round(a_eps.max(),3))
a_wage_gen = a_wage_gen + a_eps

a_X = np.hstack([a_school,a_exper,a_male])
a_X.shape

(3294,)
(3294, 1)
(3294, 1)
First five wages are set as: 
 [[5.8899994]
 [5.62     ]
 [4.87     ]
 [6.5199995]
 [6.3999996]]
Mean of error term e: -0.001
Maximum of error term e: 0.663


(3294, 3)

In [4]:
###############################################################################
#   LinReg Scikit
###############################################################################

wages_linreg_sk = LinearRegression().fit(a_X, a_wage_gen)
# Inspect the results
print("Estimated paras are")
print(wages_linreg_sk.intercept_)
print(wages_linreg_sk.coef_)
print("\n")
print("REG: R squared on data")
print(round(r2_score(a_wage_gen, wages_linreg_sk.predict(a_X)),3))

Estimated paras are
[-3.339088]
[[0.62695456 0.11917233 1.3396611 ]]


REG: R squared on data
0.973


In [5]:
###############################################################################
#   LinReg Statsmodels
###############################################################################

X = sm.add_constant(a_X)
wages_linreg_sm = sm.OLS(a_wage_gen, X)#reverse order of args
wages_linreg_sm_results = wages_linreg_sm.fit(cov_type='HAC',cov_kwds={'maxlags':1})
#menu: 'HC3', 'HAC', etc
wages_linreg_sm_results.params
# Inspect the results
print(wages_linreg_sm_results.summary())
print("\nXXX DONE XXX")

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.973
Model:                            OLS   Adj. R-squared:                  0.973
Method:                 Least Squares   F-statistic:                 3.931e+04
Date:                Thu, 19 Sep 2024   Prob (F-statistic):               0.00
Time:                        16:22:48   Log-Likelihood:                 678.56
No. Observations:                3294   AIC:                            -1349.
Df Residuals:                    3290   BIC:                            -1325.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -3.3391      0.030   -110.795      0.0

In [6]:
#same but with non-normal noise/error term, e.g. student t with shape 0 df = 10
#head of HR determines wages according to the following TOP SECRET FORMULA
a_wage_gen = -3.38 + 0.63 * a_school + 0.12 * a_exper + 1.34 * a_male  
print("First five wages are set as: \n", a_wage_gen[0:5,:])

#to hide that discrimination happens, a bit of noise is added to this equation
np.random.seed(2023)#make it reproducible
a_eps = 0.15 * np.random.standard_t(df = 10, size=school.size)#scale sd down so that similar Rsqrd
a_eps = a_eps.reshape(school.size,1)
print("Mean of error term e:", round(a_eps.mean(),3))
print("Maximum of error term e:", round(a_eps.max(),3))
a_wage_gen = a_wage_gen + a_eps

a_X = np.hstack([a_school,a_exper,a_male])
a_X.shape

First five wages are set as: 
 [[5.8899994]
 [5.62     ]
 [4.87     ]
 [6.5199995]
 [6.3999996]]
Mean of error term e: -0.0
Maximum of error term e: 0.685


(3294, 3)

In [7]:
###############################################################################
#   LinReg Statsmodels
###############################################################################

X = sm.add_constant(a_X)
wages_linreg_sm = sm.OLS(a_wage_gen, X)#reverse order of args
wages_linreg_sm_results = wages_linreg_sm.fit(cov_type='HAC',cov_kwds={'maxlags':1})
#menu: 'HC3', 'HAC', etc
wages_linreg_sm_results.params
# Inspect the results
print(wages_linreg_sm_results.summary())
print("\n Gauss Markov etc work also with non-normal error term")
print("\nXXX DONE XXX")

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.980
Model:                            OLS   Adj. R-squared:                  0.980
Method:                 Least Squares   F-statistic:                 5.220e+04
Date:                Thu, 19 Sep 2024   Prob (F-statistic):               0.00
Time:                        16:22:48   Log-Likelihood:                 1195.9
No. Observations:                3294   AIC:                            -2384.
Df Residuals:                    3290   BIC:                            -2360.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -3.3718      0.026   -127.400      0.0

In [8]:
print("DONE")

DONE
